# Eksperimen Komparasi IndoBERT vs BiLSTM untuk Analisis Sentimen Kebijakan Desil
**Objek Riset:** Kebijakan Klasifikasi Ekonomi (Desil & DTSEN) pada Media Sosial Threads, Instagram, dan Twitter (X)
**Dataset:** 3.000 data terstratifikasi (1.000 Twitter + 1.000 Threads + 1.000 Instagram)
**Model yang Dibandingkan:**
1. **IndoBERT** (`indobenchmark/indobert-base-p1`) - Pretrained Transformer (Contextual Attention)
2. **BiLSTM** (Bidirectional Long Short-Term Memory) - Recurrent Neural Network + Word Embedding

## 1. Install & Import Dependencies

In [ ]:
# Install pustaka yang dibutuhkan jika di Google Colab
!pip install -q transformers torch scikit-learn seaborn matplotlib openpyxl pandas

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

# Cek Ketersediaan GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))

# Seed Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

## 2. Load Dataset (Train, Validation, Test)

In [ ]:
# Muat data yang telah disiapkan (Train 70%, Val 15%, Test 15%)
train_df = pd.read_csv('train_set.csv')
val_df = pd.read_csv('val_set.csv')
test_df = pd.read_csv('test_set.csv')

print(f'Train set: {len(train_df)} rows')
print(f'Val set  : {len(val_df)} rows')
print(f'Test set : {len(test_df)} rows')
print('\nDistribusi Kelas pada Data Latih:')
print(train_df['sentiment'].value_counts())

# Hitung Class Weights untuk menangani data imbalanced
class_counts = train_df['label'].value_counts().sort_index().values
total_samples = len(train_df)
class_weights = total_samples / (len(class_counts) * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print('\nClass Weights Tensor:', class_weights_tensor)

## 3. MODEL 1: BiLSTM (Bidirectional LSTM)

In [ ]:
# Tokenizer Sederhana berbasis Kosakata Kata untuk BiLSTM
from collections import Counter

def build_vocab(texts, max_vocab=10000):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for w, _ in counter.most_common(max_vocab - 2):
        vocab[w] = len(vocab)
    return vocab

vocab = build_vocab(train_df['text_clean'])
print(f'Ukuran Kosakata (Vocabulary Size): {len(vocab)}')

def text_to_sequence(text, vocab, max_len=100):
    tokens = str(text).split()
    seq = [vocab.get(w, vocab['<UNK>']) for w in tokens][:max_len]
    if len(seq) < max_len:
        seq = seq + [vocab['<PAD>']] * (max_len - len(seq))
    return seq

class BiLSTMDataset(Dataset):
    def __init__(self, df, vocab, max_len=100):
        self.sequences = [text_to_sequence(t, vocab, max_len) for t in df['text_clean']]
        self.labels = df['label'].values
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_bilstm_loader = DataLoader(BiLSTMDataset(train_df, vocab), batch_size=32, shuffle=True)
val_bilstm_loader = DataLoader(BiLSTMDataset(val_df, vocab), batch_size=32, shuffle=False)
test_bilstm_loader = DataLoader(BiLSTMDataset(test_df, vocab), batch_size=32, shuffle=False)

class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=3, dropout=0.4):
        super(BiLSTMSentiment, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=2, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        # Global Max Pooling
        pooled, _ = torch.max(lstm_out, dim=1)
        out = self.dropout(pooled)
        logits = self.fc(out)
        return logits

bilstm_model = BiLSTMSentiment(len(vocab)).to(device)
criterion_bilstm = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer_bilstm = torch.optim.Adam(bilstm_model.parameters(), lr=0.001)
print(bilstm_model)

In [ ]:
# Training BiLSTM
bilstm_epochs = 15
best_bilstm_f1 = 0.0
bilstm_train_losses = []
bilstm_val_f1s = []

print('=== Memulai Training BiLSTM ===')
start_time = time.time()
for epoch in range(1, bilstm_epochs + 1):
    bilstm_model.train()
    total_loss = 0
    for x_batch, y_batch in train_bilstm_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer_bilstm.zero_grad()
        logits = bilstm_model(x_batch)
        loss = criterion_bilstm(logits, y_batch)
        loss.backward()
        optimizer_bilstm.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_bilstm_loader)
    bilstm_train_losses.append(avg_train_loss)
    
    # Evaluasi Val
    bilstm_model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for x_b, y_b in val_bilstm_loader:
            x_b = x_b.to(device)
            out = bilstm_model(x_b)
            preds = torch.argmax(out, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(y_b.numpy())
    val_f1 = f1_score(val_targets, val_preds, average='macro')
    bilstm_val_f1s.append(val_f1)
    if val_f1 > best_bilstm_f1:
        best_bilstm_f1 = val_f1
        torch.save(bilstm_model.state_dict(), 'best_bilstm_model.pt')
    print(f'Epoch {epoch:02d}/{bilstm_epochs} - Train Loss: {avg_train_loss:.4f} - Val Macro-F1: {val_f1:.4f}')

print(f'Training BiLSTM selesai dalam {time.time()-start_time:.1f} detik. Best Val F1: {best_bilstm_f1:.4f}')

## 4. MODEL 2: IndoBERT (`indobenchmark/indobert-base-p1`)

In [ ]:
MODEL_NAME = 'indobenchmark/indobert-base-p1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IndoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_bert_loader = DataLoader(IndoBERTDataset(train_df['text_clean'], train_df['label'], tokenizer), batch_size=16, shuffle=True)
val_bert_loader = DataLoader(IndoBERTDataset(val_df['text_clean'], val_df['label'], tokenizer), batch_size=16, shuffle=False)
test_bert_loader = DataLoader(IndoBERTDataset(test_df['text_clean'], test_df['label'], tokenizer), batch_size=16, shuffle=False)

bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
optimizer_bert = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
epochs_bert = 4
total_steps = len(train_bert_loader) * epochs_bert
scheduler = get_linear_schedule_with_warmup(optimizer_bert, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
criterion_bert = nn.CrossEntropyLoss(weight=class_weights_tensor)
print('IndoBERT Model & Optimizer Siap!')

In [ ]:
# Fine-Tuning IndoBERT
best_bert_f1 = 0.0
bert_train_losses = []
bert_val_f1s = []

print('=== Memulai Fine-Tuning IndoBERT ===')
start_time = time.time()
for epoch in range(1, epochs_bert + 1):
    bert_model.train()
    total_loss = 0
    for batch in train_bert_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer_bert.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion_bert(outputs.logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(bert_model.parameters(), max_norm=1.0)
        optimizer_bert.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_bert_loader)
    bert_train_losses.append(avg_loss)
    
    # Evaluasi Val
    bert_model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_bert_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(batch['label'].numpy())
    val_f1 = f1_score(val_targets, val_preds, average='macro')
    bert_val_f1s.append(val_f1)
    if val_f1 > best_bert_f1:
        best_bert_f1 = val_f1
        bert_model.save_pretrained('best_indobert_model')
        tokenizer.save_pretrained('best_indobert_model')
    print(f'Epoch {epoch:02d}/{epochs_bert} - Train Loss: {avg_loss:.4f} - Val Macro-F1: {val_f1:.4f}')

print(f'Fine-tuning IndoBERT selesai dalam {time.time()-start_time:.1f} detik. Best Val F1: {best_bert_f1:.4f}')

## 5. Evaluasi Komparatif pada Data Uji (Test Set)

In [ ]:
# 1. Evaluasi BiLSTM pada Test Set
bilstm_model.load_state_dict(torch.load('best_bilstm_model.pt'))
bilstm_model.eval()
test_preds_bilstm, test_targets = [], []
with torch.no_grad():
    for x_b, y_b in test_bilstm_loader:
        x_b = x_b.to(device)
        out = bilstm_model(x_b)
        preds = torch.argmax(out, dim=1).cpu().numpy()
        test_preds_bilstm.extend(preds)
        test_targets.extend(y_b.numpy())

# 2. Evaluasi IndoBERT pada Test Set
best_bert = AutoModelForSequenceClassification.from_pretrained('best_indobert_model').to(device)
best_bert.eval()
test_preds_bert = []
with torch.no_grad():
    for batch in test_bert_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = best_bert(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        test_preds_bert.extend(preds)

# Hitung Metrik Komparatif
target_names = ['Negatif', 'Netral', 'Positif']

metrics_data = {
    'Model': ['BiLSTM', 'IndoBERT'],
    'Accuracy': [
        accuracy_score(test_targets, test_preds_bilstm),
        accuracy_score(test_targets, test_preds_bert)
    ],
    'Macro-Precision': [
        precision_score(test_targets, test_preds_bilstm, average='macro'),
        precision_score(test_targets, test_preds_bert, average='macro')
    ],
    'Macro-Recall': [
        recall_score(test_targets, test_preds_bilstm, average='macro'),
        recall_score(test_targets, test_preds_bert, average='macro')
    ],
    'Macro-F1 (Utama)': [
        f1_score(test_targets, test_preds_bilstm, average='macro'),
        f1_score(test_targets, test_preds_bert, average='macro')
    ]
}

df_comparison = pd.DataFrame(metrics_data)
print('=== TABEL HASIL EVALUASI KOMPARATIF (TEST SET) ===')
print(df_comparison.to_string(index=False))

# Simpan Hasil Metrik ke CSV
df_comparison.to_csv('hasil_komparasi_metrik.csv', index=False)

In [ ]:
# Visualisasi Confusion Matrix Side-by-Side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_bilstm = confusion_matrix(test_targets, test_preds_bilstm)
sns.heatmap(cm_bilstm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Confusion Matrix - BiLSTM', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

cm_bert = confusion_matrix(test_targets, test_preds_bert)
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Greens', xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Confusion Matrix - IndoBERT', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.tight_layout()
plt.savefig('confusion_matrix_komparasi.png', dpi=300)
plt.show()

## 6. Evaluasi Kemampuan Generalisasi Lintas Platform (Cross-Platform)

In [ ]:
# Analisis Performa Model pada Masing-Masing Platform di Test Set
test_df['pred_bilstm'] = test_preds_bilstm
test_df['pred_indobert'] = test_preds_bert

cross_results = []
for plat in ['Twitter', 'Threads', 'Instagram']:
    sub = test_df[test_df['platform'] == plat]
    f1_bi = f1_score(sub['label'], sub['pred_bilstm'], average='macro')
    f1_bert = f1_score(sub['label'], sub['pred_indobert'], average='macro')
    acc_bi = accuracy_score(sub['label'], sub['pred_bilstm'])
    acc_bert = accuracy_score(sub['label'], sub['pred_indobert'])
    
    cross_results.append({
        'Platform': plat,
        'Jumlah Data Uji': len(sub),
        'BiLSTM Macro-F1': f1_bi,
        'IndoBERT Macro-F1': f1_bert,
        'BiLSTM Accuracy': acc_bi,
        'IndoBERT Accuracy': acc_bert
    })

df_cross = pd.DataFrame(cross_results)
print('=== HASIL EVALUASI LINTAS PLATFORM (CROSS-PLATFORM) ===')
print(df_cross.to_string(index=False))
df_cross.to_csv('evaluasi_lintas_platform.csv', index=False)